# 01. Global Forecast Error Reduction Plan

This notebook is the orchestration plan for lowering forecast error across `pm25`, `pm10`, `o3`, and `no2`.

Main objective:
- expand the station universe beyond the current local baseline
- push accepted stations above **50** before the next main retraining cycle
- build a cleaner multi-country daily dataset
- retrain offline artifacts for multiple pollutants
- promote only the models that improve rolling backtest quality

Current preferred workflow:
- manually downloaded station CSVs under `data/processed/observation_cache/` are the main expansion source
- API discovery and fetch are now optional fallback paths, not the primary route

Recommended run order:
1. This notebook for the overall action plan
2. `02_Phase1_Pipeline.ipynb` if you need to rebuild local processed inputs
3. `03_WAQI_Live_Integration.ipynb` only if you want to inspect live WAQI behavior
4. `04_Station_Expansion_Eval.ipynb` to quality-gate and evaluate the manual station CSV universe
5. `05_Forecasting.ipynb` to retrain and inspect forecasting champions after the station scope decision


## 1. Operating Targets

Primary success criteria:
- lower day-1 RMSE and MAE for all pollutants
- reduce PM2.5 MAPE toward the low-teens again
- treat O3 and NO2 with RMSE / MAE as first-class metrics because low-value days inflate MAPE
- keep app accuracy cards honest by promoting only validated champions


In [2]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
PLAN_NAME = 'global_forecast_error_reduction'
print('Project root:', PROJECT_ROOT)
print('Scripts dir:', SCRIPTS_DIR)
print('Data dir:', DATA_DIR)
print('Artifact dir:', ARTIFACT_DIR)


Project root: C:\Users\rbeyz\Desktop\AirPulse_Global
Scripts dir: C:\Users\rbeyz\Desktop\AirPulse_Global\scripts
Data dir: C:\Users\rbeyz\Desktop\AirPulse_Global\data\processed
Artifact dir: C:\Users\rbeyz\Desktop\AirPulse_Global\artifacts


## 2. Multi-Country Seed Scope

Start with a compact but diverse seed list. The goal is not every city at once; the goal is enough climates, traffic regimes, and pollution patterns to stabilize training.


In [3]:
POLLUTANTS = ['pm25', 'pm10', 'o3', 'no2']
COUNTRY_CITY_SEEDS = {
    'TR': ['Istanbul', 'Ankara', 'Izmir', 'Bursa', 'Antalya'],
    'DE': ['Berlin', 'Munich', 'Hamburg', 'Cologne', 'Frankfurt'],
    'FR': ['Paris', 'Lyon', 'Marseille', 'Lille', 'Toulouse'],
    'IT': ['Rome', 'Milan', 'Turin', 'Naples', 'Bologna'],
    'ES': ['Madrid', 'Barcelona', 'Valencia', 'Seville', 'Bilbao'],
    'GB': ['London', 'Manchester', 'Birmingham', 'Leeds', 'Glasgow'],
    'PL': ['Warsaw', 'Krakow', 'Wroclaw'],
    'NL': ['Amsterdam', 'Rotterdam', 'Utrecht'],
    'CZ': ['Prague', 'Brno'],
    'GR': ['Athens', 'Thessaloniki'],
    'RO': ['Bucharest', 'Cluj-Napoca'],
    'BG': ['Sofia', 'Plovdiv'],
    'IN': ['Delhi', 'Mumbai', 'Bengaluru'],
    'TH': ['Bangkok'],
    'KR': ['Seoul'],
}
PER_CITY_STATION_LIMIT = 8
MIN_ACCEPTED_STATION_TARGET = 50
print(json.dumps(COUNTRY_CITY_SEEDS, indent=2))


{
  "TR": [
    "Istanbul",
    "Ankara",
    "Izmir",
    "Bursa",
    "Antalya"
  ],
  "DE": [
    "Berlin",
    "Munich",
    "Hamburg",
    "Cologne",
    "Frankfurt"
  ],
  "FR": [
    "Paris",
    "Lyon",
    "Marseille",
    "Lille",
    "Toulouse"
  ],
  "IT": [
    "Rome",
    "Milan",
    "Turin",
    "Naples",
    "Bologna"
  ],
  "ES": [
    "Madrid",
    "Barcelona",
    "Valencia",
    "Seville",
    "Bilbao"
  ],
  "GB": [
    "London",
    "Manchester",
    "Birmingham",
    "Leeds",
    "Glasgow"
  ],
  "PL": [
    "Warsaw",
    "Krakow",
    "Wroclaw"
  ],
  "NL": [
    "Amsterdam",
    "Rotterdam",
    "Utrecht"
  ],
  "CZ": [
    "Prague",
    "Brno"
  ],
  "GR": [
    "Athens",
    "Thessaloniki"
  ],
  "RO": [
    "Bucharest",
    "Cluj-Napoca"
  ],
  "BG": [
    "Sofia",
    "Plovdiv"
  ],
  "IN": [
    "Delhi",
    "Mumbai",
    "Bengaluru"
  ],
  "TH": [
    "Bangkok"
  ],
  "KR": [
    "Seoul"
  ]
}


## 3. Station Discovery and Quality Gates

Do not train on every discovered station. Apply quality gates before any model retraining.


In [4]:
QUALITY_FILTERS = {
    'minimum_history_days': 180,
    'maximum_missing_ratio': 0.35,
    'minimum_recent_availability': 0.60,
    'minimum_target_coverage': 0.70,
    'maximum_gap_ratio': 0.25,
    'minimum_observations': 120,
}
print(json.dumps(QUALITY_FILTERS, indent=2))


{
  "minimum_history_days": 180,
  "maximum_missing_ratio": 0.35,
  "minimum_recent_availability": 0.6,
  "minimum_target_coverage": 0.7,
  "maximum_gap_ratio": 0.25,
  "minimum_observations": 120
}


## 4. Execution Flags

Keep every heavy step disabled until you intentionally flip it on.


In [5]:
RUN_STATION_DISCOVERY = False
RUN_HISTORY_FETCH = False
RUN_DATASET_REBUILD = False
RUN_OFFLINE_RETRAIN = False
RUN_PROMOTION_CHECK = False

DISCOVERY_NOTE = 'Turn these on one phase at a time. Review outputs after each stage.'
print(DISCOVERY_NOTE)


Turn these on one phase at a time. Review outputs after each stage.


## 5. Planned Commands

These commands are templates for the runbook. Adjust once the exact API scripts are finalized.


In [6]:
python_exe = str((PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'))
commands = {
    'station_expansion_eval': [python_exe, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', 'notebooks/04_Station_Expansion_Eval.ipynb'],
    'offline_multi_pollutant_retrain': [python_exe, 'scripts/train_offline_forecast_models.py', '--scope-mode', 'all', '--pollutants', 'pm25', 'pm10', 'o3', 'no2'],
}
for name, cmd in commands.items():
    print(name, '=>', ' '.join(cmd))


station_expansion_eval => C:\Users\rbeyz\Desktop\AirPulse_Global\.venv\Scripts\python.exe -m jupyter nbconvert --to notebook --execute notebooks/04_Station_Expansion_Eval.ipynb
offline_multi_pollutant_retrain => C:\Users\rbeyz\Desktop\AirPulse_Global\.venv\Scripts\python.exe scripts/train_offline_forecast_models.py --scope-mode all --pollutants pm25 pm10 o3 no2


## 6. Promotion Logic

Promote a new offline artifact only if all of the following are true:
- broader station scope actually exists in the processed daily dataset
- rolling evaluation improves or stays stable on RMSE and MAE
- high-pollution performance does not regress materially
- recent backtest in the app does not reject the champion


## 7. Run Order

Recommended execution order:
1. Update the seed countries and cities in this notebook.
2. Use `04_Station_Expansion_Eval.ipynb` to discover or evaluate candidate stations.
3. Rebuild the processed daily dataset once new stations are accepted.
4. Run `scripts/train_offline_forecast_models.py` for `pm25`, `pm10`, `o3`, `no2`.
5. Open `05_Forecasting.ipynb` for PM2.5 deep-dive tuning if PM2.5 still underperforms.
6. Promote artifacts only after leaderboard and recent-backtest review.
